# CURATE.AI-style dose finding: how far can 3 noisy points go?

Toy model of the idea behind CURATE.AI: fit a quadratic dose-response curve to a few noisy
measurements, then pick the dose at the top of the fitted curve.

- True response: `y(d) = 1 - ((d - 15) / 15)^2`, best dose 15 mg, peak 1.0.
- Each trial tests 3 distinct doses from 5..25 mg and adds Gaussian noise.
- Success = chosen dose within 3 mg of the true best dose. Baseline = random dose in 0..30 mg.

This is a simulation of a simplified model, not clinical evidence.

In [1]:
import numpy as np
import plotly.graph_objects as go

D_STAR, LO, HI, TOL = 15.0, 0.0, 30.0, 3.0
GRID = np.arange(5.0, 26.0, 1.0)
NOISE = np.round(np.arange(0.0, 0.41, 0.05), 2)
N = 20000
rng = np.random.default_rng(0)

def truth(d):
    return 1 - ((d - D_STAR) / 15) ** 2

def fit3(doses, y):
    """Exact quadratic through 3 points per row. doses, y: (n, 3) -> coefficients (n, 3) as a, b, c."""
    V = np.stack([doses**2, doses, np.ones_like(doses)], axis=-1)
    return np.linalg.solve(V, y[..., None])[..., 0]

def best_dose(coef):
    """Vertex if the parabola opens down (clipped to LO..HI), else the better edge."""
    a, b, c = coef.T
    with np.errstate(divide="ignore", invalid="ignore"):
        vertex = np.clip(-b / (2 * a), LO, HI)
    edge = np.where(a * LO**2 + b * LO >= a * HI**2 + b * HI, LO, HI)
    return np.where(a < 0, vertex, edge)

def trials(sd, n=N):
    doses = GRID[np.argsort(rng.random((n, GRID.size)), axis=1)[:, :3]]
    y = truth(doses) + rng.normal(0, sd, doses.shape)
    return doses, y, best_dose(fit3(doses, y))

chosen = {sd: trials(sd)[2] for sd in NOISE}
success = np.array([np.mean(np.abs(chosen[sd] - D_STAR) <= TOL) for sd in NOISE])
regret = np.array([np.mean(1 - truth(chosen[sd])) for sd in NOISE])
random_success = np.mean(np.abs(rng.uniform(LO, HI, N) - D_STAR) <= TOL)

## 1. One experiment per noise level (use the slider)

In [2]:
d = np.linspace(LO, HI, 200)
ex_rng = np.random.default_rng(3)
doses0 = np.sort(ex_rng.choice(GRID, 3, replace=False))[None]
z = ex_rng.normal(size=3)  # same noise draw, scaled by sd, so the slider is comparable

fig = go.Figure()
for sd in NOISE:
    y = truth(doses0) + sd * z
    coef = fit3(doses0, y)
    pick = float(best_dose(coef)[0])
    a, b, c = coef[0]
    fig.add_trace(go.Scatter(x=d, y=truth(d), name="true curve", line=dict(color="#888"), visible=sd == 0))
    fig.add_trace(go.Scatter(x=d, y=a*d**2 + b*d + c, name="fitted curve", line=dict(dash="dash"), visible=sd == 0))
    fig.add_trace(go.Scatter(x=doses0[0], y=y[0], mode="markers", name="measurements",
                             marker=dict(size=11), visible=sd == 0))
    fig.add_trace(go.Scatter(x=[pick], y=[truth(pick)], mode="markers", name=f"chosen dose {pick:.1f} mg",
                             marker=dict(size=14, symbol="star"), visible=sd == 0))
steps = [dict(method="update", label=f"{sd:.2f}",
              args=[dict(visible=[i // 4 == k for i in range(4 * len(NOISE))])])
         for k, sd in enumerate(NOISE)]
fig.update_layout(sliders=[dict(active=0, currentvalue=dict(prefix="noise sd: "), steps=steps)],
                  xaxis_title="dose (mg)", yaxis_title="response", yaxis_range=[-1.2, 1.6],
                  title="3 noisy measurements, fitted quadratic, chosen dose", height=480)
fig

## 2. Success rate and regret vs noise

In [3]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=NOISE, y=success, mode="lines+markers", name="fit: success rate"))
fig.add_trace(go.Scatter(x=NOISE, y=[random_success] * len(NOISE), mode="lines", name="random dose",
                         line=dict(dash="dot", color="#888")))
fig.add_trace(go.Scatter(x=NOISE, y=regret, mode="lines+markers", name="fit: mean regret", yaxis="y2"))
fig.update_layout(xaxis_title="noise sd", yaxis=dict(title="success rate (within 3 mg)", tickformat=".0%", range=[0, 1.05]),
                  yaxis2=dict(title="mean regret", overlaying="y", side="right", range=[0, 1]),
                  title=f"{N:,} trials per noise level", height=440)
fig

## 3. Where the chosen doses land (use the slider)

Piles at 0 and 30 mg are fits whose curve is flat or opens upward, so the rule falls back to an edge dose. They account for most of the regret.

In [4]:
fig = go.Figure()
for sd in NOISE:
    fig.add_trace(go.Histogram(x=chosen[sd], xbins=dict(start=LO, end=HI, size=1), visible=sd == 0,
                               marker_color="#4c78a8"))
steps = [dict(method="update", label=f"{sd:.2f}", args=[dict(visible=[i == k for i in range(len(NOISE))])])
         for k, sd in enumerate(NOISE)]
fig.add_vline(x=D_STAR, line_dash="dash", annotation_text="true best dose")
fig.update_layout(sliders=[dict(active=0, currentvalue=dict(prefix="noise sd: "), steps=steps)],
                  xaxis_title="chosen dose (mg)", yaxis_title="trials", bargap=0.05, height=440)
fig

## Two drugs: a 3D response surface

With two drugs CURATE.AI fits a second-order surface with 6 coefficients, so it needs 6 measurements
(see Figure 1 of the source paper). Same recipe: fit exactly through the points, take the best dose pair.

- True response: `y = 1 - (u/15)^2 - (v/15)^2 + 0.3 uv/225`, with `u = d1 - 12`, `v = d2 - 18`. Peak 1.0 at (12, 18) mg.
- Testable doses: 5, 10, 15, 20, 25 mg for each drug (25 combinations); each trial measures 6 of them.
- Success = chosen pair within 4 mg (straight-line distance) of (12, 18). Baseline = random pair in 0..30 mg.

In [5]:
X1 = X2 = np.arange(0.0, 30.1, 1.0)
G1, G2 = np.meshgrid(X1, X2, indexing="ij")
OPT = np.array([12.0, 18.0])
PTS = np.arange(5.0, 26.0, 5.0)
CAND = np.array([(a, b) for a in PTS for b in PTS])
TOL2 = 4.0

def truth2(x1, x2):
    u, v = x1 - OPT[0], x2 - OPT[1]
    return 1 - (u / 15) ** 2 - (v / 15) ** 2 + 0.3 * u * v / 225

def feats(x1, x2):
    return np.stack([x1**2, x1 * x2, x2**2, x1, x2, np.ones_like(x1)], axis=-1)

def fit6(x, y):
    """Exact 2nd-order surface through 6 points. x: (n, 6, 2), y: (n, 6) -> coefficients (n, 6)."""
    return (np.linalg.pinv(feats(x[..., 0], x[..., 1])) @ y[..., None])[..., 0]

F = feats(G1.ravel(), G2.ravel())  # surface features on the 31 x 31 dose grid

def trials2(sd, n=3000):
    x = CAND[np.argsort(rng.random((n, len(CAND))), axis=1)[:, :6]]
    y = truth2(x[..., 0], x[..., 1]) + rng.normal(0, sd, (n, 6))
    k = (fit6(x, y) @ F.T).argmax(1)  # best grid point of the fitted surface
    return np.stack([G1.ravel()[k], G2.ravel()[k]], axis=1)

chosen2 = {sd: trials2(sd) for sd in NOISE}
success2 = np.array([np.mean(np.linalg.norm(chosen2[sd] - OPT, axis=1) <= TOL2) for sd in NOISE])
regret2 = np.array([np.mean(1 - truth2(chosen2[sd][:, 0], chosen2[sd][:, 1])) for sd in NOISE])
random_success2 = np.mean(np.linalg.norm(rng.uniform(LO, HI, (N, 2)) - OPT, axis=1) <= TOL2)

### 4. One experiment on the 3D surface (rotate it, use the slider)

In [6]:
ex = np.random.default_rng(5)
xe = CAND[ex.choice(len(CAND), 6, replace=False)]
ze = np.random.default_rng(6).normal(size=6)
grey = [[0, "#999"], [1, "#999"]]

fig = go.Figure()
fig.add_trace(go.Surface(x=X1, y=X2, z=truth2(G1, G2).T, colorscale=grey, opacity=0.35,
                         showscale=False, name="true surface"))
fig.add_trace(go.Scatter3d(x=[OPT[0]], y=[OPT[1]], z=[1.0], mode="markers", name="true best dose",
                           marker=dict(size=6, color="black", symbol="diamond")))
for sd in NOISE:
    y = truth2(xe[:, 0], xe[:, 1]) + sd * ze
    coef = fit6(xe[None], y[None])[0]
    surf = (F @ coef).reshape(G1.shape)
    k = surf.argmax()
    p = np.array([G1.ravel()[k], G2.ravel()[k]])
    vis = sd == 0
    fig.add_trace(go.Surface(x=X1, y=X2, z=surf.T, colorscale="Viridis", opacity=0.7, showscale=False,
                             name="fitted surface", visible=vis))
    fig.add_trace(go.Scatter3d(x=xe[:, 0], y=xe[:, 1], z=y, mode="markers", name="measurements",
                               marker=dict(size=5), visible=vis))
    fig.add_trace(go.Scatter3d(x=[p[0]], y=[p[1]], z=[truth2(*p)], mode="markers",
                               name=f"chosen ({p[0]:.0f}, {p[1]:.0f}) mg",
                               marker=dict(size=7, symbol="x"), visible=vis))
n_t = 2 + 3 * len(NOISE)
steps = [dict(method="update", label=f"{sd:.2f}",
              args=[dict(visible=[i < 2 or (i - 2) // 3 == j for i in range(n_t)])])
         for j, sd in enumerate(NOISE)]
fig.update_layout(sliders=[dict(active=0, currentvalue=dict(prefix="noise sd: "), steps=steps)],
                  scene=dict(xaxis_title="drug 1 (mg)", yaxis_title="drug 2 (mg)", zaxis_title="response",
                             zaxis=dict(range=[-1.5, 1.5])),
                  title="6 noisy measurements, fitted surface, chosen dose pair", height=600)
fig

### 5. One drug vs two drugs, same noise

Tolerances differ (3 mg for one drug, 4 mg straight-line for two), so compare each curve with its own random baseline.

In [7]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=NOISE, y=success, mode="lines+markers", name="1 drug (3 points)"))
fig.add_trace(go.Scatter(x=NOISE, y=[random_success] * len(NOISE), mode="lines", name="1 drug: random",
                         line=dict(dash="dot")))
fig.add_trace(go.Scatter(x=NOISE, y=success2, mode="lines+markers", name="2 drugs (6 points)"))
fig.add_trace(go.Scatter(x=NOISE, y=[random_success2] * len(NOISE), mode="lines", name="2 drugs: random",
                         line=dict(dash="dot")))
fig.update_layout(xaxis_title="noise sd", yaxis=dict(title="success rate", tickformat=".0%", range=[0, 1.05]),
                  title="Success rate vs noise", height=440)
fig

# Part 2: published cases (real numbers, labelled)

Everything above uses a made-up curve. Below, each case says exactly what is real and what is assumed.

| Case | Real (from a source) | Assumed / simulated |
|---|---|---|
| Prostate cancer (1 patient) | 3 dose-PSA pairs and 2 CURATE.AI predictions, as quoted in the review `CURATE_AI.md` from Pantuck et al., Adv Ther 2018 (original not read) | 10% PSA measurement noise |
| Liver transplant tacrolimus | Trial numbers read from the open-access full text: Khong et al., Nat Commun 2025, PMID 40379675, NCT03527238 (target 8-10 ng/mL; large deviations 24.2% PPM vs 38.4% standard of care) | Patients, clearance drift and the standard-of-care rule are simulated; no patient-level data used |
| Multiple myeloma | Nothing yet | Deferred: no published CURATE.AI dosing data found |

Not clinical evidence. Small studies, mostly one research group.

## Case A: metastatic prostate cancer (ZEN3694 + enzalutamide 80 mg)

Real pairs: ZEN3694 36 mg gave PSA 0.91, 24 mg gave 0.68, 12 mg gave 1.60 ng/mL. Three points fit one quadratic exactly. The published CURATE.AI predictions (0.65 at 24 mg, 1.52 at 12 mg) came from an earlier profile built on 6 data triples that I do not have, so the fit below is an after-the-fact check, not a reproduction.

In [8]:
PSA_DOSE = np.array([12.0, 24.0, 36.0])
PSA_OBS = np.array([1.60, 0.68, 0.91])
PSA_PRED = {24.0: 0.65, 12.0: 1.52}  # published CURATE.AI predictions

dg = np.linspace(5, 40, 141)
a, b, c = np.polyfit(PSA_DOSE, PSA_OBS, 2)
best = -b / (2 * a)
print(f"fitted best ZEN3694 dose: {best:.1f} mg, predicted PSA {np.polyval([a, b, c], best):.2f} ng/mL")

# assumption: each PSA value has 10% multiplicative measurement noise
pr = np.random.default_rng(1)
draws = PSA_OBS * np.exp(pr.normal(0, 0.10, (2000, 3)))
co = np.array([np.polyfit(PSA_DOSE, y, 2) for y in draws])
curves = co[:, 0, None] * dg**2 + co[:, 1, None] * dg + co[:, 2, None]
lo, hi = np.percentile(curves, [5, 95], axis=0)
conc = co[:, 0] > 0
vert = -co[conc, 1] / (2 * co[conc, 0])
print(f"with 10% noise: {conc.mean():.0%} of fits open upward; best dose median {np.median(vert):.1f} mg, "
      f"5-95%: {np.percentile(vert, 5):.1f} to {np.percentile(vert, 95):.1f} mg")

fig = go.Figure()
fig.add_trace(go.Scatter(x=np.r_[dg, dg[::-1]], y=np.r_[hi, lo[::-1]], fill="toself", line=dict(width=0),
                         fillcolor="rgba(120,120,120,0.25)", name="5-95% band (assumed 10% noise)", hoverinfo="skip"))
fig.add_trace(go.Scatter(x=dg, y=a*dg**2 + b*dg + c, name="quadratic through the 3 real points"))
fig.add_trace(go.Scatter(x=PSA_DOSE, y=PSA_OBS, mode="markers", name="observed PSA (real)", marker=dict(size=12)))
fig.add_trace(go.Scatter(x=list(PSA_PRED), y=list(PSA_PRED.values()), mode="markers",
                         name="published CURATE.AI prediction", marker=dict(size=12, symbol="circle-open", line=dict(width=2))))
fig.update_layout(xaxis_title="ZEN3694 dose (mg), enzalutamide fixed at 80 mg", yaxis_title="PSA (ng/mL), lower is better",
                  title="Prostate case: PSA vs ZEN3694 dose", height=460)
fig

fitted best ZEN3694 dose: 27.6 mg, predicted PSA 0.63 ng/mL
with 10% noise: 100% of fits open upward; best dose median 27.7 mg, 5-95%: 26.1 to 29.5 mg


## Case B: liver transplant, tacrolimus (near-real simulation)

Real from Khong et al. 2025: target trough 8-10 ng/mL; a "large deviation" is more than 2 ng/mL outside the target (below 6 or above 12);
large deviations on 38.4% of days with standard of care (n = 29) vs 24.2% with PPM dosing (n = 27); any deviation on 77.3% vs 71.7% of days.
The PPM rule in the paper (their Function 2) is `dose(t+1) = TTL_target / TTL(t) * dose(t)`.

Simulated: trough = `k * dose`, where clearance `k` drifts randomly each day (log-normal steps of size `SIG`). Standard of care corrects half of
the error each day, PPM corrects all of it, doses rounded to 0.5 mg, 12 days per patient. I tuned only `SIG` so that standard of care matches 38.4%.
The PPM result is then a prediction of the model, not tuned.

In [9]:
TARGET, T_LO, T_HI = 9.0, 8.0, 10.0

def tac_paths(g, sigma, n, days=12, seed=0):
    """Trough levels (n, days). g = share of the error corrected each day (0.5 standard care, 1.0 PPM)."""
    r = np.random.default_rng(seed)
    k = np.exp(r.normal(0, 0.3, n))  # patient-level trough per mg
    dose = TARGET / k
    out = np.empty((n, days))
    for t in range(days):
        k = k * np.exp(r.normal(0, sigma, n))
        out[:, t] = k * dose
        dose = np.clip(np.round(dose * (1 + g * (TARGET / out[:, t] - 1)) * 2) / 2, 0.5, None)
    return out

large = lambda x: ((x < T_LO - 2) | (x > T_HI + 2)).mean()
anyd = lambda x: ((x < T_LO) | (x > T_HI)).mean()

grid = np.arange(0.25, 0.45, 0.01)
SIG = grid[np.argmin([abs(large(tac_paths(0.5, s, 4000)) - 0.384) for s in grid])]
soc, ppm = tac_paths(0.5, SIG, 20000, seed=1), tac_paths(1.0, SIG, 20000, seed=1)
print(f"SIG = {SIG:.2f}")
print(f"large deviations: SOC sim {large(soc):.1%} (published 38.4%), PPM sim {large(ppm):.1%} (published 24.2%)")
print(f"any deviation:    SOC sim {anyd(soc):.1%} (published 77.3%), PPM sim {anyd(ppm):.1%} (published 71.7%)")

SIG = 0.34
large deviations: SOC sim 38.2% (published 38.4%), PPM sim 31.6% (published 24.2%)
any deviation:    SOC sim 77.3% (published 77.3%), PPM sim 74.4% (published 71.7%)


In [10]:
labels = ["standard of care", "PPM"]
fig = go.Figure()
fig.add_trace(go.Bar(x=labels, y=[38.4, 24.2], name="published (Khong 2025)"))
fig.add_trace(go.Bar(x=labels, y=[large(soc) * 100, large(ppm) * 100], name="simulated"))
fig.update_layout(barmode="group", yaxis_title="% of days with a large deviation (>2 ng/mL outside 8-10)",
                  title="Tacrolimus: published vs simulated", height=420)
fig

### Example patients: same clearance drift, two dosing rules (use the slider)

In [11]:
P = 6
days = np.arange(1, 13)
fig = go.Figure()
for i in range(P):
    for name, arr in (("standard of care", soc), ("PPM", ppm)):
        fig.add_trace(go.Scatter(x=days, y=arr[i], mode="lines+markers", name=name, visible=i == 0))
fig.add_hrect(y0=T_LO, y1=T_HI, fillcolor="green", opacity=0.12, line_width=0, annotation_text="target 8-10")
fig.add_hrect(y0=T_LO - 2, y1=T_HI + 2, fillcolor="orange", opacity=0.08, line_width=0)
steps = [dict(method="update", label=str(i + 1), args=[dict(visible=[j // 2 == i for j in range(2 * P)])])
         for i in range(P)]
fig.update_layout(sliders=[dict(active=0, currentvalue=dict(prefix="simulated patient: "), steps=steps)],
                  xaxis_title="day after equilibration", yaxis_title="tacrolimus trough (ng/mL)",
                  yaxis_range=[0, 25], title="Trough level over time", height=460)
fig

## Case C: multiple myeloma

Not built yet. No published CURATE.AI dosing data found. The QPOP paper (Rashid et al., Sci Transl Med 2018, PMID 30089632) is ex vivo drug-combination screening, a different method, and I could not read its data.